In [1]:
import sqlite3

# 1. Connect to SQLite
conn = sqlite3.connect('ecommerce.db')
cursor = conn.cursor()

# 2. Enable Foreign Key support
cursor.execute("PRAGMA foreign_keys = ON;")

# 3. Create the Users Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    signup_date TEXT NOT NULL,
    country TEXT,
    traffic_source TEXT
);
""")

# 4. Create the Orders Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    order_date TEXT NOT NULL,
    order_amount REAL NOT NULL,
    status TEXT CHECK(status IN ('Completed', 'Returned', 'Cancelled')),
    FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE
);
""")

# 5. Create the Web Logs Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS web_logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    activity_date TEXT NOT NULL,
    page_views INTEGER DEFAULT 0,
    session_duration_secs INTEGER DEFAULT 0,
    FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE
);
""")

# Commit changes and close connection
conn.commit()
conn.close()

print("'ecommerce.db' created successfully with all 3 tables!")

In [2]:
# cell for adding synthetic data to the database
import random
from faker import Faker

fake = Faker()

# Connect to our local database
conn = sqlite3.connect('ecommerce.db')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

print("Generating mock data... please wait.")

# 1. Insert 1000 Users
for _ in range(1000):
    signup_date = fake.date_between(start_date="-1y", end_date="today").strftime('%Y-%m-%d')
    country = fake.country().replace("'", "")
    source = random.choice(["Organic", "Google Ads", "Facebook", "Affiliate", "Instagram", "YouTube"])
    
    cursor.execute(
        "INSERT INTO users (signup_date, country, traffic_source) VALUES (?, ?, ?)",
        (signup_date, country, source)
    )

# 2. Insert 3000 Orders
for _ in range(3000):
    user_id = random.randint(1, 1000)
    order_date = fake.date_between(start_date="-1y", end_date="today").strftime('%Y-%m-%d')
    amount = round(random.uniform(10.0, 500.0), 2)
    status = random.choice(["Completed", "Completed", "Completed", "Returned", "Cancelled"])
    
    cursor.execute(
        "INSERT INTO orders (user_id, order_date, order_amount, status) VALUES (?, ?, ?, ?)",
        (user_id, order_date, amount, status)
    )

# 3. Insert 5000 Web Logs
for _ in range(5000):
    user_id = random.randint(1, 1000)
    activity_date = fake.date_between(start_date="-1y", end_date="today").strftime('%Y-%m-%d')
    views = random.randint(1, 25)
    duration = random.randint(10, 1800)
    
    cursor.execute(
        "INSERT INTO web_logs (user_id, activity_date, page_views, session_duration_secs) VALUES (?, ?, ?, ?)",
        (user_id, activity_date, views, duration)
    )

# Commit changes to the file and close
conn.commit()
conn.close()

print("Success! Your local 'ecommerce.db' file is fully loaded with mock data.")

Generating mock data... please wait.
Success! Your local 'ecommerce.db' file is fully loaded with mock data.


In [3]:
import sqlite3
import pandas as pd

# 1. Open the connection to our database file
conn = sqlite3.connect('ecommerce.db')

# 2. Define the Master Feature Extraction Query
feature_query = """
WITH user_orders AS (
    SELECT 
        user_id,
        COUNT(order_id) AS total_orders,
        SUM(order_amount) AS total_spending,
        MAX(order_date) AS last_order_date
    FROM orders
    WHERE status = 'Completed'
    GROUP BY user_id
),
user_behavior AS (
    SELECT 
        user_id,
        SUM(page_views) AS total_page_views,
        AVG(session_duration_secs) AS avg_session_duration
    FROM web_logs
    GROUP BY user_id
)
SELECT 
    u.user_id,
    u.traffic_source,
    u.country,
    -- ML Features
    COALESCE(o.total_orders, 0) AS total_orders,
    COALESCE(o.total_spending, 0.0) AS total_spending,
    COALESCE(b.total_page_views, 0) AS total_page_views,
    COALESCE(b.avg_session_duration, 0.0) AS avg_session_duration,
    -- Target Variable: Churn (1 if no order in last 60 days, else 0)
    CASE 
        WHEN o.last_order_date IS NULL THEN 1
        WHEN julianday('2026-06-20') - julianday(o.last_order_date) > 60 THEN 1 
        ELSE 0 
    END AS churned
FROM users u
LEFT JOIN user_orders o ON u.user_id = o.user_id
LEFT JOIN user_behavior b ON u.user_id = b.user_id;
"""

# 3. Pull directly into a Pandas DataFrame
df = pd.read_sql_query(feature_query, conn)
conn.close()

# 4. Display the results
print("Phase 2 Complete: Dataset extracted successfully!")
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 Rows of your ML Dataset:")
print(df.head())

Phase 2 Complete: Dataset extracted successfully!
Dataset Dimensions: 3000 rows, 8 columns

First 5 Rows of your ML Dataset:
   user_id traffic_source               country  total_orders  total_spending  \
0        1        Organic             Australia             4         1020.56   
1        2        Organic               Jamaica             7         1284.41   
2        3      Affiliate                Norway            11         2592.69   
3        4        Organic  Netherlands Antilles             7         1635.99   
4        5      Affiliate          Burkina Faso             5         1250.80   

   total_page_views  avg_session_duration  churned  
0               182           1045.125000        1  
1               102            537.888889        0  
2               234            638.526316        0  
3               161            853.928571        0  
4                97            874.000000        1  


In [4]:
#Cell for feature engineering and model training
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Fetch the data from our SQLite database
conn = sqlite3.connect('ecommerce.db')
feature_query = """
WITH user_orders AS (
    SELECT user_id, COUNT(order_id) AS total_orders, SUM(order_amount) AS total_spending, MAX(order_date) AS last_order_date
    FROM orders WHERE status = 'Completed' GROUP BY user_id
),
user_behavior AS (
    SELECT user_id, SUM(page_views) AS total_page_views, AVG(session_duration_secs) AS avg_session_duration
    FROM web_logs GROUP BY user_id
)
SELECT 
    u.traffic_source, u.country,
    COALESCE(o.total_orders, 0) AS total_orders, COALESCE(o.total_spending, 0.0) AS total_spending,
    COALESCE(b.total_page_views, 0) AS total_page_views, COALESCE(b.avg_session_duration, 0.0) AS avg_session_duration,
    CASE 
        WHEN o.last_order_date IS NULL THEN 1
        WHEN julianday('2026-06-22') - julianday(o.last_order_date) > 60 THEN 1 
        ELSE 0 
    END AS churned
FROM users u
LEFT JOIN user_orders o ON u.user_id = o.user_id
LEFT JOIN user_behavior b ON u.user_id = b.user_id;
"""
df = pd.read_sql_query(feature_query, conn)
conn.close()

# 2. Separate features (X) from the target label (y)
X = df.drop(columns=['churned'])
y = df['churned']

# 3. Split data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Define Preprocessing Steps for Numeric vs. Categorical Columns
numeric_features = ['total_orders', 'total_spending', 'total_page_views', 'avg_session_duration']
categorical_features = ['traffic_source', 'country']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),               # Standardize numerical features
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # Convert text to binary 1s/0s
    ]
)

# 5. Bundle preprocessing and the ML model together into a clean Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

# 6. Train the model!
print("Training the Random Forest model...")
pipeline.fit(X_train, y_train)
print("Model training complete!")

# 7. Evaluate the performance on unseen test data
y_pred = pipeline.predict(X_test)
print("\n--- Model Accuracy Score ---")
print(f"{accuracy_score(y_test, y_pred) * 100:.2f}%")

print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

Training the Random Forest model...
Model training complete!

--- Model Accuracy Score ---
88.00%

--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       0.65      0.80      0.72       114
           1       0.95      0.90      0.92       486

    accuracy                           0.88       600
   macro avg       0.80      0.85      0.82       600
weighted avg       0.89      0.88      0.88       600



In [5]:
#Cell for feature engineering(2) and model training[Using grid search]
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Connect and run another Feature Engineering Query
conn = sqlite3.connect('ecommerce.db')
advanced_query = """
WITH user_orders AS (
    SELECT 
        user_id, 
        COUNT(order_id) AS total_orders, 
        SUM(order_amount) AS total_spending, 
        MAX(order_date) AS last_order_date
    FROM orders WHERE status = 'Completed' GROUP BY user_id
),
user_behavior AS (
    SELECT 
        user_id, 
        SUM(page_views) AS total_page_views, 
        SUM(session_duration_secs) AS total_duration
    FROM web_logs GROUP BY user_id
)
SELECT 
    u.traffic_source, u.country,
    COALESCE(o.total_orders, 0) AS total_orders, 
    COALESCE(o.total_spending, 0.0) AS total_spending,
    COALESCE(b.total_page_views, 0) AS total_page_views,
    
    -- New Feature 1: Average Order Value (AOV)
    CASE WHEN COALESCE(o.total_orders, 0) > 0 THEN o.total_spending / o.total_orders ELSE 0.0 END AS avg_order_value,
    
    -- New Feature 2: Page Views per Second (Engagement Intensity)
    CASE WHEN COALESCE(b.total_duration, 0) > 0 THEN CAST(b.total_page_views AS REAL) / b.total_duration ELSE 0.0 END AS views_per_sec,
    
    -- New Feature 3: Tenure (How many days since they signed up)
    julianday('2026-06-22') - julianday(u.signup_date) AS account_tenure_days,
    
    CASE 
        WHEN o.last_order_date IS NULL THEN 1
        WHEN julianday('2026-06-22') - julianday(o.last_order_date) > 60 THEN 1 
        ELSE 0 
    END AS churned
FROM users u
LEFT JOIN user_orders o ON u.user_id = o.user_id
LEFT JOIN user_behavior b ON u.user_id = b.user_id;
"""
df = pd.read_sql_query(advanced_query, conn)
conn.close()

X = df.drop(columns=['churned'])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Setup Preprocessor
numeric_features = ['total_orders', 'total_spending', 'total_page_views', 'avg_order_value', 'views_per_sec', 'account_tenure_days']
categorical_features = ['traffic_source', 'country']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# 3. Create Pipeline with class_weight='balanced' to handle imbalance
base_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# 4. Define Hyperparameter Grid for Tuning
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, 20, None]
}

# 5. Run Grid Search to find the absolute best model configurations based on F1-score
print(" Searching for optimal hyperparameters via Grid Search...")
grid_search = GridSearchCV(base_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f" Best Hyperparameters Found: {grid_search.best_params_}")

# 6. Evaluate the newly optimized model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- Optimized Classification Report ---")
print(classification_report(y_test, y_pred))

 Searching for optimal hyperparameters via Grid Search...
 Best Hyperparameters Found: {'classifier__max_depth': None, 'classifier__n_estimators': 100}

--- Optimized Classification Report ---
              precision    recall  f1-score   support

           0       0.64      0.79      0.71       114
           1       0.95      0.90      0.92       486

    accuracy                           0.88       600
   macro avg       0.79      0.84      0.81       600
weighted avg       0.89      0.88      0.88       600

